# Phase 4a - Sentiment analysis with Spark MLlib

**Goal:** read a review's **text** and predict whether it is **negative, neutral or positive**.
The answer key comes from the star rating (1-2 = negative, 3 = neutral, 4-5 = positive, made in Phase 2).

**Plan**

| Step | What | Why |
|---|---|---|
| 1 | Split 80% train / 20% test, keeping the class mix the same | fair test on reviews the model never saw |
| 2 | Class weights | 75.8% of reviews are positive - without weights the model ignores neutral |
| 3 | Features: TF-IDF on single words **and word pairs** | word pairs keep "not good" together |
| 4 | Tune Logistic Regression with 3-fold cross-validation | pick the best regularisation |
| 5 | Train and compare 4 models + a baseline | Logistic Regression, Naive Bayes, Linear SVM |
| 6 | Evaluate with **macro F1**, per-class F1, confusion matrix | accuracy alone is misleading here |
| 7 | Look inside the model: most important words per class | explain results in the viva |
| 8 | Save the model and score every review | the fake-review module will use these scores |

**Baseline to beat (from the synopsis):** always predict "positive" -> about 75.8% accuracy but a very low macro F1.

Run the cells **top to bottom**. Expect about **25-40 minutes** on free Colab (most of it is model training).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/amazon_reviews'
SAMPLE = '1pct'
LOCAL = '/content/parquet'
FIG_DIR = f'{PROJECT}/results/figures'
OUT_DIR = f'{PROJECT}/results/sentiment'
MODEL_DIR = f'{PROJECT}/models'
SEED = 42

In [ ]:
import os
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip -q install pyspark==3.5.3

In [ ]:
!mkdir -p {LOCAL} {FIG_DIR} {OUT_DIR} {MODEL_DIR}
!cp -r {PROJECT}/parquet/reviews_clean_{SAMPLE} {LOCAL}/

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('Phase4a-Sentiment')
         .master('local[*]')
         .config('spark.driver.memory', '9g')
         .config('spark.sql.shuffle.partitions', '16')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

reviews = spark.read.parquet(f'{LOCAL}/reviews_clean_{SAMPLE}')
data = (reviews
        .filter(F.size('tokens') > 0)
        .select('review_id', 'text', 'rating', 'sentiment', 'label', 'tokens')
        .cache())
print(f'reviews for modelling: {data.count():,}')
data.groupBy('label', 'sentiment').count().orderBy('label').show()

### Helpers: chart style and metrics

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BLUE, ORANGE, RED, GREY = '#2a78d6', '#eb6834', '#e34948', '#9a9893'
TEXT, MUTED, GRID = '#0b0b0b', '#52514e', '#e6e5e0'
CLASS_NAMES = ['negative', 'neutral', 'positive']          # label 0, 1, 2
CLASS_COLORS = {'negative': RED, 'neutral': GREY, 'positive': BLUE}

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10,
    'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'axes.ymargin': 0.12,
    'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': TEXT, 'axes.titlelocation': 'left',
    'legend.frameon': False,
})

def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight', facecolor='white')
    plt.show()

def summarize(cm, name, train_seconds=0.0):
    '''Metrics from a 3x3 confusion matrix (rows = true class, columns = predicted class).'''
    m = np.asarray(cm, dtype=float)
    tp = np.diag(m)
    precision = tp / np.maximum(m.sum(axis=0), 1)
    recall = tp / np.maximum(m.sum(axis=1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    support = m.sum(axis=1)
    row = {'model': name,
           'accuracy': tp.sum() / m.sum(),
           'macro_f1': f1.mean(),
           'weighted_f1': (f1 * support).sum() / support.sum()}
    for i, c in enumerate(CLASS_NAMES):
        row[f'precision_{c}'] = precision[i]
        row[f'recall_{c}'] = recall[i]
        row[f'f1_{c}'] = f1[i]
    row['train_seconds'] = round(train_seconds, 1)
    return row

In [ ]:
def confusion(pred_df):
    '''3x3 confusion matrix (pandas) from a Spark DataFrame with label and prediction.'''
    counts = pred_df.groupBy('label', 'prediction').count().toPandas()
    cm = (counts.pivot(index='label', columns='prediction', values='count')
                .reindex(index=[0.0, 1.0, 2.0], columns=[0.0, 1.0, 2.0]).fillna(0))
    cm.index, cm.columns = CLASS_NAMES, CLASS_NAMES
    return cm

## 1. Train / test split (stratified)

`sampleBy` takes 80% of **each class** for training, so train and test have the same
positive / neutral / negative mix. Everything not in train becomes test.

In [ ]:
train = data.sampleBy('label', fractions={0.0: 0.8, 1.0: 0.8, 2.0: 0.8}, seed=SEED)
test = data.join(train.select('review_id'), 'review_id', 'left_anti')

mix = (train.groupBy('sentiment').count().withColumnRenamed('count', 'train')
       .join(test.groupBy('sentiment').count().withColumnRenamed('count', 'test'), 'sentiment')
       .toPandas())
mix['train_pct'] = (100 * mix['train'] / mix['train'].sum()).round(1)
mix['test_pct'] = (100 * mix['test'] / mix['test'].sum()).round(1)
mix

## 2. Class weights

Weight = total reviews / (3 x reviews in that class). A rare class (neutral) gets a **big**
weight, the common class (positive) a **small** one, so mistakes on every class count about equally.

In [ ]:
counts = {r['label']: r['count'] for r in train.groupBy('label').count().collect()}
total = sum(counts.values())
class_weight = {lbl: total / (3 * n) for lbl, n in counts.items()}
for lbl in sorted(class_weight):
    print(f'{CLASS_NAMES[int(lbl)]:<9} reviews {counts[lbl]:>8,}   weight {class_weight[lbl]:.2f}')

weight_expr = F.create_map(*[x for kv in class_weight.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])
train = train.withColumn('weight', weight_expr[F.col('label')])

## 3. Features: TF-IDF on words and word pairs

| Stage | Example for "not good battery" |
|---|---|
| tokens (from Phase 2) | `[not, good, battery]` |
| **NGram(2)** - word pairs | `[not good, good battery]` |
| combine both | `[not, good, battery, not good, good battery]` |
| **CountVectorizer** - count each term | keeps the 100,000 most common terms seen in 5+ reviews |
| **IDF** - down-weight terms that appear everywhere | "product" gets a low weight, "not good" a high one |

We use `CountVectorizer` instead of `HashingTF` because it keeps the **vocabulary**, so in
step 7 we can print which words the model relies on. (`HashingTF` is faster on the full
dataset because it needs no vocabulary - mention it as the option for scale.)

The feature pipeline is **fitted on train only**, so nothing from the test set leaks into training.

In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import NGram, SQLTransformer, CountVectorizer, IDF

ngram = NGram(n=2, inputCol='tokens', outputCol='bigrams')
merge = SQLTransformer(statement='SELECT *, concat(tokens, bigrams) AS terms FROM __THIS__')
counter = CountVectorizer(inputCol='terms', outputCol='tf', vocabSize=100_000, minDF=5)
idf = IDF(inputCol='tf', outputCol='features', minDocFreq=5)

t0 = time.time()
feature_model = Pipeline(stages=[ngram, merge, counter, idf]).fit(train)
print(f'feature pipeline fitted in {time.time() - t0:.0f}s')

vocab = feature_model.stages[2].vocabulary
print(f'vocabulary size: {len(vocab):,}')
print('example word pairs:', [t for t in vocab if ' ' in t][:15])

train_f = feature_model.transform(train).select('review_id', 'label', 'weight', 'features').cache()
test_f = feature_model.transform(test).select('review_id', 'label', 'features').cache()
print(f'train: {train_f.count():,}   test: {test_f.count():,}')

## 4. Tune Logistic Regression with 3-fold cross-validation

`regParam` controls how much the model is held back from memorising the training data.
We try 3 values with **3-fold cross-validation** (each value is trained 3 times on different
parts of the data). To keep it fast on Colab, tuning runs on a 25% slice of the training set;
the final models are then trained on the **full** training set.

In [ ]:
from pyspark.ml.classification import LogisticRegression, NaiveBayes, LinearSVC, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

lr_tune = LogisticRegression(featuresCol='features', labelCol='label', weightCol='weight', maxIter=50)
grid = ParamGridBuilder().addGrid(lr_tune.regParam, [0.001, 0.01, 0.1]).build()
cross_val = CrossValidator(estimator=lr_tune, estimatorParamMaps=grid,
                           evaluator=MulticlassClassificationEvaluator(labelCol='label', metricName='f1'),
                           numFolds=3, seed=SEED, parallelism=2)

t0 = time.time()
cv_model = cross_val.fit(train_f.sample(fraction=0.25, seed=SEED))
cv_table = pd.DataFrame({'regParam': [p[lr_tune.regParam] for p in grid],
                         'cv_f1': np.round(cv_model.avgMetrics, 4)})
BEST_REG = float(cv_table.loc[cv_table['cv_f1'].idxmax(), 'regParam'])
cv_table.to_csv(f'{OUT_DIR}/cv_results_{SAMPLE}.csv', index=False)
print(f'cross-validation took {time.time() - t0:.0f}s  ->  best regParam = {BEST_REG}')
cv_table

## 5. Train and compare the models

| Model | Idea in one line |
|---|---|
| Baseline | always say "positive" |
| Logistic Regression, **no** class weights | to show what imbalance does |
| Logistic Regression, class weights | learns a weight for every word / word pair per class |
| Naive Bayes, class weights | counts how often each word appears in each class (very fast) |
| Linear SVM (one-vs-rest), class weights | finds the widest gap between classes; Spark's SVM is 2-class, so one-vs-rest trains one SVM per class |

In [ ]:
results, matrices, models = [], {}, {}

def run(name, estimator):
    t0 = time.time()
    model = estimator.fit(train_f)
    seconds = time.time() - t0
    pred = model.transform(test_f).select('label', 'prediction')
    cm = confusion(pred)
    row = summarize(cm.values, name, seconds)
    results.append(row); matrices[name] = cm; models[name] = model
    print(f"{name:<42} accuracy {row['accuracy']:.3f}   macro F1 {row['macro_f1']:.3f}   ({seconds:.0f}s)")

# baseline: no training, always predict positive (label 2)
base_cm = confusion(test_f.select('label').withColumn('prediction', F.lit(2.0)))
results.append(summarize(base_cm.values, 'Baseline: always positive'))
matrices['Baseline: always positive'] = base_cm
print(f"{'Baseline: always positive':<42} accuracy {results[0]['accuracy']:.3f}   macro F1 {results[0]['macro_f1']:.3f}")

run('Logistic Regression (no weights)',
    LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=BEST_REG))
run('Logistic Regression (class weights)',
    LogisticRegression(featuresCol='features', labelCol='label', weightCol='weight', maxIter=100, regParam=BEST_REG))
run('Naive Bayes (class weights)',
    NaiveBayes(featuresCol='features', labelCol='label', weightCol='weight', modelType='multinomial', smoothing=1.0))
run('Linear SVM one-vs-rest (class weights)',
    OneVsRest(classifier=LinearSVC(maxIter=50, regParam=BEST_REG),
              featuresCol='features', labelCol='label', weightCol='weight'))

In [ ]:
comparison = pd.DataFrame(results)
comparison.to_csv(f'{OUT_DIR}/model_comparison_{SAMPLE}.csv', index=False)
BEST = comparison.iloc[1:].sort_values('macro_f1', ascending=False)['model'].iloc[0]
print('Best model by macro F1:', BEST)

show_cols = ['model', 'accuracy', 'macro_f1', 'weighted_f1', 'f1_negative', 'f1_neutral', 'f1_positive', 'train_seconds']
comparison[show_cols].round(3)

## 6. Results

**Why two panels?** The baseline already scores high **accuracy** by always saying "positive" -
but its **macro F1** (the average F1 of the three classes) is low, because it gets negative and
neutral completely wrong. Macro F1 is the fair score when classes are imbalanced.

In [ ]:
order = comparison['model'].tolist()[::-1]
data_plot = comparison.set_index('model').loc[order]
colors = [GREY if m.startswith('Baseline') else (ORANGE if m == BEST else BLUE) for m in order]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, metric, title in [(axes[0], 'accuracy', 'Accuracy'), (axes[1], 'macro_f1', 'Macro F1 (fair for imbalance)')]:
    bars = ax.barh(order, data_plot[metric], height=0.6, color=colors, edgecolor='white', linewidth=1.5)
    ax.bar_label(bars, labels=[f'{v:.3f}' for v in data_plot[metric]], padding=3, color=TEXT, fontsize=9)
    ax.set_xlim(0, 1.08)
    ax.grid(axis='x', visible=True)
    ax.grid(axis='y', visible=False)
    ax.set_title(title)
fig.suptitle('Sentiment models vs baseline (orange = best, grey = baseline)', x=0.01, ha='left',
             fontsize=12, fontweight='bold', color=TEXT)
save_fig(fig, '4a_model_comparison')

### Effect of class weights

Recall = of all reviews that really are neutral (or negative), how many did the model find?

In [ ]:
pair = comparison.set_index('model').loc[['Logistic Regression (no weights)', 'Logistic Regression (class weights)']]
fig, ax = plt.subplots(figsize=(8, 3.8))
width = 0.38
for i, (name, color, label) in enumerate([('Logistic Regression (no weights)', GREY, 'no weights'),
                                          ('Logistic Regression (class weights)', BLUE, 'class weights')]):
    vals = [pair.loc[name, f'recall_{c}'] for c in CLASS_NAMES]
    bars = ax.bar([x + (i - 0.5) * width for x in range(3)], vals, width=width,
                  color=color, edgecolor='white', linewidth=1.5, label=label)
    ax.bar_label(bars, labels=[f'{v:.2f}' for v in vals], padding=3, color=TEXT, fontsize=9)
ax.set_xticks(range(3), CLASS_NAMES)
ax.set_ylim(0, 1.1)
ax.set_ylabel('recall')
ax.set_title('Class weights help the model find neutral and negative reviews')
ax.legend(loc='upper left', ncols=2)
save_fig(fig, '4a_class_weights_recall')

### Confusion matrix of the best model

Each row is the **true** class and adds up to 100%. The diagonal is "correct".
Expect most mistakes around **neutral** - a 3-star review often sounds partly positive and partly negative.

In [ ]:
cm = matrices[BEST]
cm.to_csv(f'{OUT_DIR}/confusion_matrix_best_{SAMPLE}.csv')
cm_pct = cm.div(cm.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(5.8, 4.8))
ax.imshow(cm_pct.values, cmap='Blues', vmin=0, vmax=100)
ax.grid(False)
for i in range(3):
    for j in range(3):
        v = cm_pct.values[i, j]
        ax.text(j, i, f'{v:.1f}%\n({int(cm.values[i, j]):,})', ha='center', va='center',
                color='white' if v > 55 else TEXT, fontsize=9)
ax.set_xticks(range(3), CLASS_NAMES)
ax.set_yticks(range(3), CLASS_NAMES)
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title(f'Confusion matrix - {BEST}', fontsize=10)
save_fig(fig, '4a_confusion_matrix')

## 7. Which words does the model rely on?

Logistic Regression learns one weight per term for each class. The terms with the **highest**
weight for a class push a review towards that class. Look for word pairs like *"not work"* or
*"waste money"* - they show why we kept "not" and added word pairs.

In [ ]:
lr_model = models['Logistic Regression (class weights)']
coef = lr_model.coefficientMatrix.toArray()          # shape: 3 classes x vocabulary
top_terms = {}
for i, c in enumerate(CLASS_NAMES):
    idx = np.argsort(coef[i])[::-1][:15]
    top_terms[c] = pd.DataFrame({'term': [vocab[j] for j in idx], 'weight': coef[i][idx].round(3)})
    top_terms[c].to_csv(f'{OUT_DIR}/top_terms_{c}_{SAMPLE}.csv', index=False)
pd.concat(top_terms, axis=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, c in zip(axes, CLASS_NAMES):
    pdf = top_terms[c].iloc[::-1]
    ax.barh(pdf['term'], pdf['weight'], height=0.6, color=CLASS_COLORS[c], edgecolor='white', linewidth=1.5)
    ax.grid(axis='x', visible=True)
    ax.grid(axis='y', visible=False)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(4))
    ax.set_title(f'Strongest terms: {c}')
save_fig(fig, '4a_top_terms')

### Mistakes worth reading

Real reviews the best model got wrong. Reading 10-20 of these is the best way to write the
"error analysis" paragraph of the report (sarcasm, mixed opinions, wrong star rating...).

In [ ]:
pd.set_option('display.max_colwidth', 200)
best_pred = models[BEST].transform(test_f).select('review_id', 'prediction')
mistakes = (best_pred.join(test.select('review_id', 'label', 'rating', 'sentiment', 'text'), 'review_id')
            .filter(F.col('prediction') != F.col('label'))
            .withColumn('predicted', F.element_at(F.array(*[F.lit(c) for c in CLASS_NAMES]),
                                                  F.col('prediction').cast('int') + 1))
            .select('rating', 'sentiment', 'predicted', 'text')
            .orderBy(F.rand(SEED)).limit(15).toPandas())
mistakes.to_csv(f'{OUT_DIR}/mistakes_sample_{SAMPLE}.csv', index=False)
mistakes

## 8. Save the model and score every review

We save **feature pipeline + weighted Logistic Regression** as one Spark `PipelineModel`.
Logistic Regression is used for scoring (even if another model won by a little) because it
gives a **probability** for each class - the fake-review module needs that, e.g. to find a
5-star review whose text reads as clearly negative.

In [ ]:
scoring_model = PipelineModel(stages=feature_model.stages + [models['Logistic Regression (class weights)']])
local_model = f'/content/models/sentiment_lr_{SAMPLE}'
scoring_model.write().overwrite().save(local_model)
!rm -rf {MODEL_DIR}/sentiment_lr_{SAMPLE} && cp -r {local_model} {MODEL_DIR}/
print('model saved to', f'{MODEL_DIR}/sentiment_lr_{SAMPLE}')

In [ ]:
from pyspark.ml.functions import vector_to_array

scores = (scoring_model.transform(reviews.filter(F.size('tokens') > 0))
          .withColumn('p', vector_to_array('probability'))
          .select('review_id', 'rating', 'label',
                  F.col('prediction').alias('predicted_label'),
                  F.round(F.col('p')[0], 4).alias('p_negative'),
                  F.round(F.col('p')[1], 4).alias('p_neutral'),
                  F.round(F.col('p')[2], 4).alias('p_positive')))

scores.write.mode('overwrite').parquet(f'{LOCAL}/sentiment_scores_{SAMPLE}')
!rm -rf {PROJECT}/parquet/sentiment_scores_{SAMPLE} && cp -r {LOCAL}/sentiment_scores_{SAMPLE} {PROJECT}/parquet/

saved = spark.read.parquet(f'{PROJECT}/parquet/sentiment_scores_{SAMPLE}')
print(f'scored reviews saved: {saved.count():,}')
saved.show(5)

In [ ]:
# preview for the fake-review module: 4-5 star reviews whose text the model reads as clearly negative
mismatch = (saved.filter((F.col('rating') >= 4) & (F.col('p_negative') >= 0.8))
                 .join(reviews.select('review_id', 'text'), 'review_id'))
print(f'4-5 star reviews with clearly negative text: {mismatch.count():,}')
mismatch.select('rating', 'p_negative', 'text').limit(10).toPandas()

## Done - what you have now

| File (in Drive) | What |
|---|---|
| `results/sentiment/model_comparison_1pct.csv` | the main results table for the report |
| `results/sentiment/cv_results_1pct.csv` | cross-validation results |
| `results/sentiment/confusion_matrix_best_1pct.csv` | confusion matrix |
| `results/sentiment/top_terms_*.csv` | strongest words per class |
| `results/sentiment/mistakes_sample_1pct.csv` | examples for error analysis |
| `results/figures/4a_*.png` | 4 charts |
| `models/sentiment_lr_1pct/` | the saved model |
| `parquet/sentiment_scores_1pct/` | sentiment probabilities for every review |

**Send back:** the comparison table (section 5), the CV table (section 4) and the
"4-5 star reviews with clearly negative text" count.

**Next:** Phase 4b - fake review detection.